# LightRAG × HotpotQA（distractor）最小实验

这个 Notebook 以共享语料库方式在 HotpotQA 上运行 LightRAG，并输出可断点续跑的逐题结果与 Answer EM/F1。它参考了所给 Notebook 的**数据采样、持久化和评测接口**，但不复用其中 GraphRAG 或单题向量检索的方法实现。

实验设定：从 `validation` 中固定随机抽取 hard + bridge 样本；汇总这些样本全部候选段落并按标题去重；用该共享语料建立一次 LightRAG 索引；再逐题查询。它是 *closed-corpus distractor* 实验，不等同于 HotpotQA FullWiki 排行榜设定。

> 提醒：实体/关系抽取会产生大量 LLM 调用。先以 `NUM_QUESTIONS=25` 做冒烟测试，确认后再改为 200。

In [ ]:
# 在 Colab/Jupyter 的新环境中执行一次。完成后请重启内核。
%pip install -q "lightrag-hku" datasets pandas tqdm openai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.8/89.8 kB 3.4 MB/s eta 0:00:00


In [ ]:
import asyncio
from functools import partial
import json
import os
import re
import shutil
import string
import time
from collections import Counter
from getpass import getpass
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

# 不把密钥写入 Notebook 或磁盘。也可在运行前由终端设置 OPENAI_API_KEY。
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OpenAI API key: ').strip()
if not os.environ['OPENAI_API_KEY']:
    raise ValueError('OPENAI_API_KEY 不能为空')

# 先跑小样本，确认成本、依赖与权限正常后再设为 200。
NUM_QUESTIONS = 200
RANDOM_SEED = 42
CHAT_MODEL = 'gpt-4o-mini'
ANSWER_NORMALIZER_MODEL = 'gpt-4o-mini'  # 仅负责把冗长回答压缩为最终答案
EMBEDDING_MODEL = 'text-embedding-3-small'  # openai_embed 的默认模型
QUERY_MODES = ['hybrid']  # 完整比较可改为 ['naive', 'local', 'global', 'hybrid']
PROJECT_ROOT = Path('hotpot_lightrag')
DATA_DIR = PROJECT_ROOT / 'experiment_data'
RAG_DIR = PROJECT_ROOT / 'lightrag_store'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RAG_DIR.mkdir(parents=True, exist_ok=True)
print({'NUM_QUESTIONS': NUM_QUESTIONS, 'modes': QUERY_MODES, 'root': str(PROJECT_ROOT)})


OpenAI API key: ··········
{'NUM_QUESTIONS': 200, 'modes': ['hybrid'], 'root': 'hotpot_lightrag'}


## 1. 固定评测题目并建立共享语料

不能为每个问题只插入该题的十个候选段落，否则没有测试共享知识库检索能力。这里模仿参考 GraphRAG Notebook：选定问题后，把它们的候选段落合并、按标题去重并建立一次索引。标题被保留在正文中，方便图谱抽取跨文档实体关系。

In [ ]:
QUESTIONS_PATH = DATA_DIR / 'questions.json'
CORPUS_PATH = DATA_DIR / 'corpus.jsonl'

def prepare_hotpot_subset(num_questions=NUM_QUESTIONS, seed=RANDOM_SEED):
    dataset = load_dataset('hotpotqa/hotpot_qa', 'distractor', split='validation')
    subset = (dataset
              .filter(lambda row: row['level'] == 'hard' and row['type'] == 'bridge')
              .shuffle(seed=seed)
              .select(range(min(num_questions, len(dataset)))))

    docs_by_title, questions = {}, []
    for row in subset:
        questions.append({
            'id': row['id'],
            'question': row['question'],
            'answer': row['answer'],
            'supporting_facts': row['supporting_facts'],
        })
        for title, sentences in zip(row['context']['title'], row['context']['sentences']):
            text = ' '.join(sentence.strip() for sentence in sentences if sentence.strip()).strip()
            # 同标题有多个版本时，保留更完整的一份。
            if len(text) > len(docs_by_title.get(title, '')):
                docs_by_title[title] = text

    corpus = [
        {'doc_id': f'hotpot-{i:06d}', 'title': title, 'text': text}
        for i, (title, text) in enumerate(sorted(docs_by_title.items()))
    ]
    QUESTIONS_PATH.write_text(json.dumps(questions, ensure_ascii=False, indent=2), encoding='utf-8')
    with CORPUS_PATH.open('w', encoding='utf-8') as handle:
        for doc in corpus:
            handle.write(json.dumps(doc, ensure_ascii=False) + '\n')
    return questions, corpus

questions, corpus = prepare_hotpot_subset()
print(f'questions={len(questions)}, unique candidate documents={len(corpus)}')
print(corpus[0]['title'])
print(corpus[0]['text'][:300])


README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

...

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7405 [00:00<?, ? examples/s]

questions=200, unique candidate documents=1986
(I Can't Make It) Another Day
"(I Can't Make It) Another Day" is a song by American recording artist Michael Jackson featuring Lenny Kravitz, released on the posthumous album "Michael" which has sold 6.5 million globally. Initially leaked as a 90-second snippet, it was referred to as "Another Day". Shortly after the leak, singer


## 2. 初始化并建立 LightRAG 索引

官方 SDK 适合此类研究/评测用途。`naive` 是纯 chunk 向量检索，`local`、`global`、`hybrid` 会使用知识图谱；因此在相同索引与同一题集上切换模式即可形成内部对照。若更改采样、模型、embedding 或 chunk 参数，请删除 `hotpot_lightrag/lightrag_store` 后重新索引；本 Notebook 不会自动删除已有索引。

In [ ]:
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete, openai_embed
from lightrag.utils import EmbeddingFunc

INDEX_META_PATH = RAG_DIR / 'hotpot_index_meta.json'

def render_document(doc):
    return f"Document ID: {doc['doc_id']}\nTitle: {doc['title']}\nContent: {doc['text']}"

# 每个 Wikipedia 标题段落作为独立文档插入，保留来源边界与可追溯性。
# 不要把整个 corpus 拼成一个长字符串；否则 LightRAG 会把它记为 1 个 full document。
corpus_texts = [render_document(doc) for doc in corpus]
corpus_ids = [doc['doc_id'] for doc in corpus]

# 保留 LightRAG 所需的 embedding_dim/max_token_size 元数据，同时固定模型。
# 若切换为 text-embedding-3-large，须相应改为 EMBEDDING_DIM = 3072 并重建索引。
EMBEDDING_DIM = 1536
embedding_func = EmbeddingFunc(
    embedding_dim=EMBEDDING_DIM,
    max_token_size=8192,
    func=partial(openai_embed.func, model=EMBEDDING_MODEL),
)

async def build_or_load_rag():
    rag = LightRAG(
        working_dir=str(RAG_DIR),
        llm_model_func=openai_complete,
        llm_model_name=CHAT_MODEL,
        embedding_func=embedding_func,
        chunk_token_size=800,
        chunk_overlap_token_size=100,
        llm_model_max_async=4,
        addon_params={'language': 'English'},
    )
    await rag.initialize_storages()

    expected_meta = {
        'num_questions': NUM_QUESTIONS, 'seed': RANDOM_SEED,
        'chat_model': CHAT_MODEL, 'embedding_model': EMBEDDING_MODEL,
        'num_documents': len(corpus), 'insertion_mode': 'one-document-per-title',
    }
    if INDEX_META_PATH.exists():
        actual_meta = json.loads(INDEX_META_PATH.read_text(encoding='utf-8'))
        if actual_meta != expected_meta:
            raise RuntimeError(
                '当前索引与配置不一致。请删除 lightrag_store 后重新执行本节，以免混用索引。'
            )
        print('检测到与当前配置一致的索引，直接加载。')
    else:
        started = time.time()
        await rag.ainsert(corpus_texts, ids=corpus_ids)
        INDEX_META_PATH.write_text(json.dumps(expected_meta, indent=2), encoding='utf-8')
        print(f'索引完成：{(time.time() - started) / 60:.1f} 分钟')
    return rag

rag = await build_or_load_rag()


流式输出内容被截断，只能显示最后 5000 行内容。
INFO: Upserting relation VDB: `Cherokee Campground->Tagg Run Campground`
INFO: Upserting relation VDB: `Common Well->Toland`
INFO: Upserting relation VDB: `Duplex Houses->Toland`
INFO: Upserting relation VDB: `Toland->Toland Community`
INFO: Upserting relation VDB: `Clay Bank->Toland`
INFO: Upserting relation VDB: `One-Room School House->Toland`
INFO: Upserting relation VDB: `Clay Bank->Sand Pit Operation`
INFO: Upserting relation VDB: `Families of Toland->Toland`
INFO: Phase 2 relation processing completed for hotpot-001798: edges=21 added_entities=0
INFO: Completed merging: 26 entities, 0 extra entities, 21 relations
INFO: [] entities flush: embedding 17 vectors in 2 batch(es) (batch_num=10)
INFO: [] relationships flush: embedding 21 vectors in 3 batch(es) (batch_num=10)
INFO: [] chunks flush: embedding 1 vectors in 1 batch(es) (batch_num=10)
INFO: [] Writing graph with 11568 nodes, 12111 edges
INFO: In memory DB persist to disk
INFO: Completed processing f

索引完成：197.8 分钟


In [ ]:
# 冒烟测试：先检查一个问题的输出，再进行批量评测。
example = questions[0]
response = await rag.aquery(
    example['question'],
    param=QueryParam(
        mode='hybrid', top_k=20, chunk_top_k=8, enable_rerank=False,
        response_type='Return only one short final answer, without explanation or citations.',
    ),
)
print('Question:', example['question'])
print('Gold:', example['answer'])
print('LightRAG:', response)


INFO: keyword LLM func: 4 new workers initialized (Timeouts: Func: 240s, Worker: 480s, Health Check: 495s)
INFO:  == LLM cache == saving: hybrid:keywords:038c91fbc4db4094dff2fad3d4e53fda
INFO: Query nodes: young woman, yellow shawl (top_k:20, cosine:0.2)
INFO: Local query: 20 entites, 32 relations
INFO: Query edges: Dutch painter, domestic interior scenes, middle class life, Louvre Paris (top_k:20, cosine:0.2)
INFO: Global query: 26 entites, 20 relations
INFO: Raw search results: 44 entities, 49 relations, 0 vector chunks
INFO: After truncation: 44 entities, 49 relations
INFO: Selecting 38 from 38 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 49 relations
INFO: Round-robin merged chunks: 38 -> 38 (deduplicated 0)
INFO: Final context: 44 entities, 49 relations, 8 chunks
INFO: Final chunks S+F/O: E5/1 E3/2 E3/3 E5/4 E4/5 E7/6 E2/7 E2/8
INFO: query LLM func: 4 new workers initialized (Timeouts: Func: 240s, Worker: 480s, Health Check: 495

Question: What Dutch painter specialized in domestic interior scenes of middle class life and has a painting of a young woman dressed in a yellow shawl in the Louvre, Paris?
Gold: Johannes Vermeer
LightRAG: The Dutch painter who specialized in domestic interior scenes of middle-class life and has a painting of a young woman dressed in a yellow shawl in the Louvre, Paris, is Johannes Vermeer. This painting is titled "The Lacemaker."

### References
- [1] The Lacemaker (Vermeer)
- [2] Johannes Vermeer


## 3. 批量查询（支持断点续跑）

每种查询模式分别写入 JSONL；已成功完成的 `(mode, id)` 不会重复请求。中断后直接重跑此单元格即可。

In [ ]:
PREDICTIONS_PATH = DATA_DIR / 'lightrag_predictions.jsonl'

def completed_pairs(path):
    if not path.exists():
        return set()
    pairs = set()
    for line in path.read_text(encoding='utf-8').splitlines():
        try:
            row = json.loads(line)
            if row.get('status') == 'ok':
                pairs.add((row['mode'], row['id']))
        except json.JSONDecodeError:
            pass
    return pairs

async def answer_question(question, mode):
    result = await rag.aquery(
        question,
        param=QueryParam(
            mode=mode, top_k=20, chunk_top_k=8, enable_rerank=False,
            response_type='Return only one short final answer, without explanation or citations.',
        ),
    )
    return str(result).strip()

done = completed_pairs(PREDICTIONS_PATH)
for mode in QUERY_MODES:
    for item in tqdm(questions, desc=f'LightRAG {mode}'):
        if (mode, item['id']) in done:
            continue
        started = time.time()
        record = {
            'id': item['id'], 'mode': mode, 'question': item['question'],
            'gold_answer': item['answer'], 'seconds': None,
        }
        try:
            record['prediction'] = await answer_question(item['question'], mode)
            record['status'] = 'ok'
        except Exception as error:
            record['prediction'] = ''
            record['status'] = 'error'
            record['error'] = repr(error)
        record['seconds'] = round(time.time() - started, 3)
        with PREDICTIONS_PATH.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')
        print(f"[{mode}] {item['id']} | {record['status']} | {record['prediction']!r}")

print('Saved:', PREDICTIONS_PATH)


LightRAG hybrid:   0%|          | 0/200 [00:00<?, ?it/s]

INFO: Query nodes: young woman, yellow shawl (top_k:20, cosine:0.2)
INFO: Local query: 20 entites, 32 relations
INFO: Query edges: Dutch painter, domestic interior scenes, middle class life, Louvre Paris (top_k:20, cosine:0.2)
INFO: Global query: 26 entites, 20 relations
INFO: Raw search results: 44 entities, 49 relations, 0 vector chunks
INFO: After truncation: 44 entities, 49 relations
INFO: Selecting 38 from 38 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 49 relations
INFO: Round-robin merged chunks: 38 -> 38 (deduplicated 0)
INFO: Final context: 44 entities, 49 relations, 8 chunks
INFO: Final chunks S+F/O: E5/1 E3/2 E3/3 E5/4 E4/5 E7/6 E2/7 E2/8
INFO:  == LLM cache == Query cache hit, using cached response as query result


[hybrid] 5ac47d7355429919431739b3 | ok | 'The Dutch painter who specialized in domestic interior scenes of middle-class life and has a painting of a young woman dressed in a yellow shawl in the Louvre, Paris, is Johannes Vermeer. This painting is titled "The Lacemaker."\n\n### References\n- [1] The Lacemaker (Vermeer)\n- [2] Johannes Vermeer'


INFO:  == LLM cache == saving: hybrid:keywords:f47b4b82a06adc03123acc23800c727b
INFO: Query nodes: Aughton Park railway station, Northern Line, Liverpool (top_k:20, cosine:0.2)
INFO: Local query: 20 entites, 40 relations
INFO: Query edges: transportation, railway stations, geography (top_k:20, cosine:0.2)
INFO: Global query: 33 entites, 20 relations
INFO: Raw search results: 50 entities, 57 relations, 0 vector chunks
INFO: After truncation: 50 entities, 57 relations
INFO: Selecting 40 from 40 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 57 relations
INFO: Round-robin merged chunks: 40 -> 40 (deduplicated 0)
INFO: Final context: 50 entities, 57 relations, 8 chunks
INFO: Final chunks S+F/O: E6/1 E7/2 E6/3 E3/4 E2/5 E3/6 E2/7 E1/8
INFO:  == LLM cache == saving: hybrid:query:5fbf736295258c4cc7dd5e7e739bcadd


[hybrid] 5ab7f27e5542991d322237d6 | ok | 'Aughton Park railway station is located approximately 11½ miles (19 km) northeast of Liverpool Central. \n\n### References\n- [1] Aughton Park railway station'


...

[hybrid] 5a8750945542996432c57264 | ok | 'The actress who was killed off along with Sonny Valentine is Gemma Merna, who portrayed Carmel McQueen in "Hollyoaks." She is known for her comedic personality and has become notable for her "dumb blonde" appearance. In terms of her role, she played a character that was involved in significant storylines, including her sacrifice to save her cousin. \n\nHowever, there is no information in the provided context suggesting that Gemma Merna had a character specifically identified as a teacher in the soap opera. \n\n### References\n\n- [1] Sonny Valentine\n- [2] Carmel McQueen'


INFO:  == LLM cache == saving: hybrid:keywords:4c634eb195fa96d575d5a27dee12ff58
INFO: Query nodes: Michael Polcino, Dominic (top_k:20, cosine:0.2)
INFO: Local query: 20 entites, 34 relations
INFO: Query edges: animated sitcom, setting (top_k:20, cosine:0.2)
INFO: Global query: 29 entites, 20 relations
INFO: Raw search results: 49 entities, 54 relations, 0 vector chunks
INFO: After truncation: 49 entities, 54 relations
INFO: Selecting 61 from 61 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 54 relations
INFO: Round-robin merged chunks: 61 -> 61 (deduplicated 0)
INFO: Final context: 49 entities, 54 relations, 8 chunks
INFO: Final chunks S+F/O: E6/1 E4/2 E4/3 E3/4 E3/5 E3/6 E2/7 E3/8
INFO:  == LLM cache == saving: hybrid:query:7a53a8299dac2ae1f258e78852bc8d2f


[hybrid] 5a8f8e4d55429918e830d260 | ok | 'The setting for the animated sitcom that Dominic Polcino works on, which is "Family Guy," is the fictional city of Quahog, Rhode Island. \n\n### References\n* [1] Family Guy\n* [2] Dominic Polcino'


INFO:  == LLM cache == saving: hybrid:keywords:085f852d7e180c8bf6f42ddda55e0771
INFO: Query nodes: 25 miles, south, Groom Lake (top_k:20, cosine:0.2)
INFO: Local query: 20 entites, 43 relations
INFO: Query edges: geography, locations, Groom Lake (top_k:20, cosine:0.2)
INFO: Global query: 20 entites, 20 relations
INFO: Raw search results: 29 entities, 46 relations, 0 vector chunks
INFO: After truncation: 29 entities, 46 relations
INFO: Selecting 19 from 19 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 46 relations
INFO: Round-robin merged chunks: 19 -> 19 (deduplicated 0)
INFO: Final context: 29 entities, 46 relations, 8 chunks
INFO: Final chunks S+F/O: E8/1 E5/2 E6/3 E5/4 E2/5 E4/6 E1/7 E2/8
INFO:  == LLM cache == saving: hybrid:query:44a4be5306fd99ef07dbb77bd82fc6f5


[hybrid] 5adfff0755429925eb1afbce | ok | '25 miles south of Groom Lake is the town of Rachel, Nevada. \n\n### References\n\n- [1] Rachel, Nevada'
Saved: hotpot_lightrag/experiment_data/lightrag_predictions.jsonl


## 4. 将长回答规范化为短答案（推荐执行）

LightRAG 的回答有时会忽略 `response_type` 并输出完整解释。这个单元格使用独立的低成本模型，只从原始回答中抽取最终答案；它不允许使用外部知识来修正答案。原始回答始终保留在 `raw_prediction`，短答案写到新的 JSONL，因而可以安全重跑或审计。

> 该后处理会对每条成功预测增加一次短模型调用。

In [ ]:
from openai import AsyncOpenAI

NORMALIZED_PREDICTIONS_PATH = DATA_DIR / 'lightrag_short_predictions.jsonl'
normalizer_client = AsyncOpenAI(api_key=os.environ['OPENAI_API_KEY'])

async def extract_final_answer(question, long_answer):
    prompt = f"""You are a strict answer-formatting function.

Question: {question}
Model answer: {long_answer}

Extract only the shortest final answer stated by the model answer.
Do not explain, cite evidence, add words, or use outside knowledge to correct it.
If the model answer explicitly says the answer cannot be determined, return exactly: Cannot be determined from the provided context.
Return only the answer text.
"""
    completion = await normalizer_client.chat.completions.create(
        model=ANSWER_NORMALIZER_MODEL,
        messages=[
            {'role': 'system', 'content': 'Return only the final short answer.'},
            {'role': 'user', 'content': prompt},
        ],
        temperature=0,
        max_tokens=32,
    )
    return completion.choices[0].message.content.strip()

raw_records = [json.loads(line) for line in PREDICTIONS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
already_normalized = set()
if NORMALIZED_PREDICTIONS_PATH.exists():
    for line in NORMALIZED_PREDICTIONS_PATH.read_text(encoding='utf-8').splitlines():
        try:
            previous = json.loads(line)
            if previous.get('normalization_status') == 'ok':
                already_normalized.add((previous.get('mode'), previous.get('id')))
        except json.JSONDecodeError:
            pass

for raw in tqdm(raw_records, desc='Normalizing answers'):
    key = (raw.get('mode'), raw.get('id'))
    if raw.get('status') != 'ok' or key in already_normalized:
        continue
    started = time.time()
    record = {**raw, 'raw_prediction': raw.get('prediction', '')}
    try:
        record['prediction'] = await extract_final_answer(record['question'], record['raw_prediction'])
        record['normalization_status'] = 'ok'
    except Exception as error:
        record['prediction'] = record['raw_prediction']
        record['normalization_status'] = 'error'
        record['normalization_error'] = repr(error)
    record['normalization_seconds'] = round(time.time() - started, 3)
    with NORMALIZED_PREDICTIONS_PATH.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')

print('Saved:', NORMALIZED_PREDICTIONS_PATH)


Normalizing answers:   0%|          | 0/200 [00:00<?, ?it/s]

Saved: hotpot_lightrag/experiment_data/lightrag_short_predictions.jsonl


## 5. Answer EM / F1 与结果导出

这部分沿用 HotpotQA/SQuAD 风格的答案规范化。LightRAG 的 SDK 输出不是原始 `(title, sentence_id)` supporting facts，因此此版本只将 Answer EM/F1 作为主要指标；空 `sp` 不能用于解读 Support 或 Joint F1。

In [ ]:
def normalize_answer(text):
    text = str(text).lower()
    text = ''.join(char for char in text if char not in string.punctuation)
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    return ' '.join(text.split())

def exact_match(prediction, gold):
    return float(normalize_answer(prediction) == normalize_answer(gold))

def token_f1(prediction, gold):
    prediction_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()
    if not prediction_tokens or not gold_tokens:
        return float(prediction_tokens == gold_tokens)
    overlap = sum((Counter(prediction_tokens) & Counter(gold_tokens)).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(prediction_tokens)
    recall = overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

# 优先评测规范化后的短答案；若尚未运行上一单元格，则回退到原始长回答。
EVALUATED_PREDICTIONS_PATH = (NORMALIZED_PREDICTIONS_PATH
                              if NORMALIZED_PREDICTIONS_PATH.exists()
                              else PREDICTIONS_PATH)
records = [json.loads(line) for line in EVALUATED_PREDICTIONS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
# 断点续跑可能留下同一题的旧失败记录；只保留每个 (mode, id) 的最新记录。
latest_records = {}
for record in records:
    latest_records[(record.get('mode'), record.get('id'))] = record
records = list(latest_records.values())
df = pd.DataFrame(records)
ok = df[df['status'].eq('ok')].copy()
ok['EM'] = ok.apply(lambda row: exact_match(row['prediction'], row['gold_answer']), axis=1)
ok['F1'] = ok.apply(lambda row: token_f1(row['prediction'], row['gold_answer']), axis=1)
summary = (ok.groupby('mode', as_index=False)
           .agg(questions=('id', 'count'), answer_em=('EM', 'mean'),
                answer_f1=('F1', 'mean'), avg_seconds=('seconds', 'mean'))
           .sort_values('mode'))
display(summary)
display(ok[['mode', 'question', 'gold_answer', 'prediction', 'EM', 'F1', 'seconds']].head(20))

EVALUATION_PATH = DATA_DIR / 'lightrag_evaluation.json'
EVALUATION_PATH.write_text(
    json.dumps({'config': {'num_questions': NUM_QUESTIONS, 'seed': RANDOM_SEED,
                           'chat_model': CHAT_MODEL, 'embedding_model': EMBEDDING_MODEL},
                'summary': summary.to_dict(orient='records'),
                'records': records}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('Evaluated:', EVALUATED_PREDICTIONS_PATH)
print('Saved:', EVALUATION_PATH)


,mode,questions,answer_em,answer_f1,avg_seconds
0,hybrid,200,0.5,0.691456,3.37258


,mode,question,gold_answer,prediction,EM,F1,seconds
0,hybrid,What Dutch painter specialized in domestic int...,Johannes Vermeer,Johannes Vermeer,1.0,1.000000,0.370
1,hybrid,Aughton Park railway station is on a branch of...,12+3/4 mi northeast of Liverpool,approximately 11½ miles (19 km) northeast of L...,0.0,0.428571,2.916
2,hybrid,Who is the son of the author of the short stor...,Benjamin Hale Cheever,Benjamin Cheever,0.0,0.800000,2.819
3,hybrid,One of the members of Dru Hill is nicknamed af...,Woody Woodpecker,Woody Woodpecker,1.0,1.000000,3.797
4,hybrid,"How many records did the singer who sang ""Four...",140 million,Cannot be determined from the provided context.,0.0,0.000000,2.937
5,hybrid,The director of a film about Wong Fei-hung als...,"""The 36th Chamber of Shaolin""","""Warriors Two.""",0.0,0.000000,2.752
6,hybrid,Both Bishop Carroll Catholic High School and K...,Wichita,"Wichita, Kansas.",0.0,0.666667,3.111
7,hybrid,Luke Rockhold defeated the MMA fighter who was...,Anderson Silva,Anderson Silva,1.0,1.000000,4.706
8,hybrid,Who wrote the novel that the movie directed by...,Stephen King,Stephen King,1.0,1.000000,3.220
9,hybrid,What is the name of the daughter of the Irish ...,Chloë Alexandra Adele Emily Agnew,Chloë Agnew,0.0,0.571429,2.969


Evaluated: hotpot_lightrag/experiment_data/lightrag_short_predictions.jsonl
Saved: hotpot_lightrag/experiment_data/lightrag_evaluation.json


In [ ]:
# 可选：导出为 HotpotQA 预测文件。sp 暂为空，故不要将其 Joint F1 作为 LightRAG 的真实表现。
def export_hotpot_answers(frame, mode):
    selected = frame[(frame['mode'] == mode) & (frame['status'] == 'ok')]
    payload = {
        'answer': dict(zip(selected['id'], selected['prediction'])),
        'sp': {question_id: [] for question_id in selected['id']},
    }
    target = DATA_DIR / f'lightrag_{mode}_hotpot_predictions.json'
    target.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return target

for mode in QUERY_MODES:
    print(export_hotpot_answers(df, mode))


hotpot_lightrag/experiment_data/lightrag_hybrid_hotpot_predictions.json


## 下一步：使其成为更严格的 HotpotQA 实验

- 将 `QUERY_MODES` 改为 `['naive', 'local', 'global', 'hybrid']`，比较同一索引下的模式差异。
- 将 `NUM_QUESTIONS` 改为 200；保持 seed、模型、chunk 参数不变。
- 对回答后返回的 source chunk 做标题/句子切分匹配，生成 `supporting_facts`，再运行官方 `hotpot_evaluate_v1.py` 计算 Support/Joint 指标。
- 若目标是可与 FullWiki 工作横向比较的结果，需要替换为完整 Wikipedia 检索语料，而不是此处的 distractor 候选段落池。